# 17 Embedding 与输出层权重绑定如何实现？

## 面试回答主线

权重绑定让输入 embedding 矩阵 $E\in\mathbb R^{V\times d}$ 同时作为输出词表投影的转置：$\text{logits}=hE^T$。它减少参数并让输入/输出词语空间共享几何结构，但要求 hidden dim 与 embedding dim 可兼容；不兼容时需 adapter。面试中要指出绑定的是同一 Parameter，不是复制相同数值的两个矩阵，否则优化器状态和内存都不会共享。实验用六个客服关键词预测下一意图，比较 untied 与 tied 的参数量和梯度对象，并演示维度不匹配时应加投影。

**核心公式：** Untied 有 $E\in\mathbb R^{V\times d}$ 与 $W_o\in\mathbb R^{d\times V}$；tied 使用 $W_o=E^T$，参数从 $2Vd$ 降为 $Vd$（不计 bias/adapter）。

下面按真实案例、基线、手写机制、结果表和失败修复组织回答；所有数据都是可复现的教学实验。


## 真实案例

场景是客服与账户安全系统中的六条脱敏离线事件。字段包含工单文本、有效 token 数和风险标签；它们模拟真实的数据结构，但样本极小，只用于观察公式和状态变化。


In [1]:
import math  # 导入数学函数以实现训练与掩码公式。
import warnings  # 导入警告控制模块保持输出干净。
warnings.filterwarnings('ignore', message='The pynvml package is deprecated')  # 屏蔽环境依赖的非教学弃用提示。
import torch  # 导入张量计算和自动微分能力。
import torch.nn as nn  # 导入模块基类以手写网络结构。
torch.manual_seed(29)  # 固定随机种子使教学输出可复现。
torch.set_num_threads(1)  # 限制小实验 CPU 线程数。
samples = [  # 构造六条脱敏客服对话作为真实语义样本。
    {'id': 'C01', 'text': '支付重复扣款，申请退款', 'tokens': 6, 'risk': 1},  # 资金风险工单。
    {'id': 'C02', 'text': '收不到登录验证码', 'tokens': 2, 'risk': 0},  # 登录支持工单。
    {'id': 'C03', 'text': '账户有陌生转账记录', 'tokens': 5, 'risk': 1},  # 账户安全工单。
    {'id': 'C04', 'text': '修改订单收货地址', 'tokens': 3, 'risk': 0},  # 售后咨询工单。
    {'id': 'C05', 'text': '银行卡盗刷需要冻结', 'tokens': 7, 'risk': 1},  # 高优先级安全工单。
    {'id': 'C06', 'text': '更正发票抬头信息', 'tokens': 4, 'risk': 0},  # 账单服务工单。
]  # 结束教学数据定义。
features = torch.tensor([[1.0, 0.0, 1.0], [0.0, 1.0, 0.0], [1.0, 0.0, 0.0], [0.0, 0.0, 1.0], [1.0, 1.0, 0.0], [0.0, 1.0, 1.0]])  # 构造三维可解释特征。
labels = torch.tensor([1, 0, 1, 0, 1, 0])  # 构造风险分类标签。
print('教学实验：六条脱敏离线客服事件，只验证机制，不代表线上收益。')  # 声明数据边界。
for row in samples:  # 逐条展示真实语义输入。
    print(f"{row['id']} | token={row['tokens']} | risk={row['risk']} | {row['text']}")  # 输出样本字段。
print(f'特征形状={tuple(features.shape)}，标签={labels.tolist()}')  # 输出张量形状。


教学实验：六条脱敏离线客服事件，只验证机制，不代表线上收益。
C01 | token=6 | risk=1 | 支付重复扣款，申请退款
C02 | token=2 | risk=0 | 收不到登录验证码
C03 | token=5 | risk=1 | 账户有陌生转账记录
C04 | token=3 | risk=0 | 修改订单收货地址
C05 | token=7 | risk=1 | 银行卡盗刷需要冻结
C06 | token=4 | risk=0 | 更正发票抬头信息
特征形状=(6, 3)，标签=[1, 0, 1, 0, 1, 0]


## Baseline / 基线

先在同一批六条事件上运行最简单方案。基线不是稻草人，它提供固定的输入、口径和可比较指标。


In [2]:
vocabulary = ['退款', '盗刷', '登录', '地址', '发票', '冻结']  # 定义六个与客服意图对应的词表项。
untied_embedding = nn.Parameter(torch.randn(6, 3) * 0.2)  # 创建输入 embedding 矩阵。
untied_head = nn.Parameter(torch.randn(3, 6) * 0.2)  # 创建独立输出投影矩阵。
hidden_state = untied_embedding[torch.tensor([0, 1, 2, 3, 4, 5])].mean(dim=0, keepdim=True)  # 用六个关键词聚合出一个教学 hidden state。
untied_logits = hidden_state @ untied_head  # 用独立输出头生成词表 logits。
baseline_metric = untied_embedding.numel() + untied_head.numel()  # 统计 untied 参数量。
print(f'Untied：参数量={baseline_metric}，logits 形状={tuple(untied_logits.shape)}，embedding 与 head 是不同对象={untied_embedding is not untied_head}')  # 展示基线内存和对象关系。


Untied：参数量=36，logits 形状=(1, 6)，embedding 与 head 是不同对象=True


## 手写核心实现与中间量

代码保留关键分子分母、mask、梯度、参数组或重算路径，而不让 Trainer 或高层框架隐藏面试问题本身。


In [3]:
class TiedLanguageHead(nn.Module):  # 手写共享输入输出参数的语言模型头。
    def __init__(self, vocab_size, width):  # 接收词表大小和隐藏宽度。
        super().__init__()  # 初始化模块父类。
        self.embedding = nn.Parameter(torch.randn(vocab_size, width) * 0.2)  # 创建唯一的共享词表矩阵。
    def forward(self, token_ids):  # 显式实现查表、聚合和 tied 输出投影。
        hidden = self.embedding[token_ids].mean(dim=0, keepdim=True)  # 查表并聚合关键词表示。
        return hidden @ self.embedding.T  # 使用同一 Parameter 的转置产生 logits。
tied_model = TiedLanguageHead(6, 3)  # 建立共享参数模型。
tied_logits = tied_model(torch.tensor([0, 1, 2, 3, 4, 5]))  # 运行真实 tied 前向传播。
core_metric = tied_model.embedding.numel()  # 统计共享后唯一矩阵的参数量。
print(f'Tied：参数量={core_metric}，logits 形状={tuple(tied_logits.shape)}，共享对象 id={id(tied_model.embedding)}')  # 展示共享而非复制。


Tied：参数量=18，logits 形状=(1, 6)，共享对象 id=131403452938128


In [4]:
comparison_rows = [('Baseline', float(baseline_metric)), ('核心机制', float(core_metric))]  # 建立同一口径的结果表。
for name, metric in comparison_rows:  # 逐行输出结果。
    print(f'{name:<8} | 指标={metric:.6f}')  # 展示可读数值对照。


Baseline | 指标=36.000000
核心机制     | 指标=18.000000


## 结果解读

基线和核心输出只在本受控案例中比较。生产需要确认词表、sharding 和 checkpoint key 一致；tensor parallel 下还要确保 embedding 与 lm-head 的分片布局可以复用。 观察结果时应关注中间量是否符合公式，而不是把六条样本上的数字宣传为线上收益。

## 失败案例

下一个单元故意破坏关键假设，并用实现修复证明该假设为何必要。


In [5]:
hidden_width = 4  # 构造与 embedding 宽度不匹配的 hidden state 维度。
embedding_width = 3  # 记录词表 embedding 的维度。
failure_metric = int(hidden_width != embedding_width)  # 检查直接 tie 是否会发生维度不兼容。
adapter = torch.randn(hidden_width, embedding_width) * 0.1  # 创建把 hidden 投影回 embedding 空间的 adapter。
fixed_hidden = torch.randn(1, hidden_width) @ adapter  # 执行 adapter 投影得到可绑定的维度。
fix_metric = int(fixed_hidden.shape[-1] == embedding_width)  # 检查修复后维度兼容。
print(f'失败：hidden={hidden_width} 与 embedding={embedding_width} 不可直接 tie；修复：adapter 后维度={fixed_hidden.shape[-1]}')  # 展示 shape 契约。


失败：hidden=4 与 embedding=3 不可直接 tie；修复：adapter 后维度=3


## 工程取舍、常见坑与延伸追问

**工程取舍：** 生产需要确认词表、sharding 和 checkpoint key 一致；tensor parallel 下还要确保 embedding 与 lm-head 的分片布局可以复用。

**常见坑：** 创建两个数值相等的矩阵假装 tie，或在 hidden dim 不等于 embedding dim 时强行转置导致 shape 错误。

**延伸追问：** 为何有些模型 tie 输入/输出而有些不 tie？量化、词表扩展和 LoRA 会如何影响共享参数？

## 生产差距

本 Notebook 在 CPU/FP32 下处理 6 条离线事件，省略了真实 token packing、分布式同步、混合精度、checkpoint、隐私治理、监控告警和灰度回滚。生产版本必须替换为受审计的数据管道与系统级指标。


In [6]:
assert baseline_metric == 36  # 验证 untied 使用两块 6x3 矩阵。
assert core_metric == 18  # 验证 tied 只保留一块 6x3 矩阵。
assert failure_metric == 1  # 验证维度不一致会阻止直接绑定。
assert fix_metric == 1  # 验证 adapter 恢复了维度兼容。
